# Dataset e Modelagem EnergIAI V2

Este notebook executa a análise exploratória, a divisão estratificada, a validação multivariada, o treinamento, a avaliação e a preparação do pipeline do Dataset V2.

> O resultado mede a capacidade do modelo de reproduzir padrões da base sintética sob as condições testadas.

O dataset é integralmente sintético e os resultados não devem ser interpretados como desempenho em dados reais.

## 1. Configuração e reprodutibilidade

Configuração de caminhos, seed, bibliotecas e parâmetros globais.

In [ ]:
"""Configuração inicial do notebook EnergIAI V2."""

from __future__ import annotations

import hashlib
import json
import platform
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import scipy
import sklearn

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_PATH = (
    PROJECT_ROOT
    / "data-science"
    / "data"
    / "dataset_energiai_v2.csv"
)
METADATA_PATH = (
    PROJECT_ROOT
    / "data-science"
    / "data"
    / "dataset_energiai_v2.metadata.json"
)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Dataset:", DATA_PATH)


## 2. Carregamento e validação do artefato

O CSV será carregado localmente nesta branch. Após merge e tag, a referência deverá ser substituída por commit ou tag imutável.

In [ ]:
"""Carregamento do CSV e dos metadados."""

dataset = pd.read_csv(DATA_PATH)
metadata = json.loads(
    METADATA_PATH.read_text(encoding="utf-8")
)

dataset.head()


### 2.1. Verificação de hash, schema e quantidade

In [ ]:
"""Verificações iniciais do artefato versionado."""

calculated_hash = hashlib.sha256(
    DATA_PATH.read_bytes()
).hexdigest()

assert calculated_hash == metadata["sha256"]
assert len(dataset) == metadata["record_count"]
assert list(dataset.columns) == metadata["schema"]

print("SHA-256 validado:", calculated_hash)
print("Registros:", len(dataset))
print("Colunas:", len(dataset.columns))


## 3. Relatório de qualidade

Esta etapa reutiliza a validação programática do artefato candidato para verificar schema, quantidade de registros, valores nulos, valores não finitos, duplicatas, domínios, limites, distribuições e quotas auditáveis.

O relatório não avalia desempenho preditivo. Ele comprova somente a consistência interna do dataset sintético versionado.

In [ ]:
"""Executa o relatório de qualidade do Dataset V2."""

import sys

SOURCE_PATH = PROJECT_ROOT / "data-science" / "src"

if str(SOURCE_PATH) not in sys.path:
    sys.path.insert(0, str(SOURCE_PATH))

import dataset_artifact


quality_summary = dataset_artifact.validate_final_dataset(
    dataset
)

quality_indicators = pd.Series(
    {
        "registros": quality_summary["record_count"],
        "colunas": quality_summary["column_count"],
        "duplicatas_integrais": quality_summary[
            "duplicate_count"
        ],
        "duplicatas_features": quality_summary[
            "feature_duplicate_count"
        ],
        "valores_nulos": quality_summary["null_count"],
        "valores_nao_finitos": quality_summary[
            "non_finite_count"
        ],
        "casos_fronteira": quality_summary[
            "boundary_count"
        ],
        "casos_raros": quality_summary["rare_count"],
        "outliers_plausiveis": quality_summary[
            "outlier_count"
        ],
    },
    name="valor",
)

class_distribution = pd.Series(
    quality_summary["class_distribution"],
    name="quantidade",
).rename_axis("categoria")

property_distribution = pd.Series(
    quality_summary["property_distribution"],
    name="quantidade",
).rename_axis("tipo_imovel")

scenario_distribution = pd.Series(
    quality_summary["scenario_distribution"],
    name="quantidade",
).rename_axis("tipo_cenario")

assert quality_summary["record_count"] == 5_000
assert quality_summary["column_count"] == 12
assert quality_summary["duplicate_count"] == 0
assert quality_summary["feature_duplicate_count"] == 0
assert quality_summary["null_count"] == 0
assert quality_summary["non_finite_count"] == 0
assert quality_summary["boundary_count"] == 150
assert quality_summary["rare_count"] == 250
assert quality_summary["outlier_count"] == 150

print("Lote:", quality_summary["generation_lot"])
print("Relatório de qualidade aprovado.")

display(quality_indicators.to_frame())
display(class_distribution.to_frame())
display(property_distribution.to_frame())
display(scenario_distribution.to_frame())


## 4. Auditoria contratual dos 5.000 registros

Esta seção audita o artefato integral sem orientar seleção ou ajuste de
modelos. São verificadas as classes oficiais, os seis tipos de imóvel, os
cenários, as quotas, os casos de fronteira, os raros e os outliers plausíveis.

A EDA orientada à modelagem, as correlações, o PCA e o K-Means são executados
somente depois da divisão e exclusivamente sobre o conjunto de treino.


### 4.1. Frequência das categorias


In [ ]:
"""Analisa a frequência das categorias do Dataset EnergIAI V2."""

EXPECTED_CATEGORIES = [
    "EFICIENTE",
    "MODERADO",
    "INEFICIENTE",
]

if "categoria" not in dataset.columns:
    raise KeyError(
        "A coluna categoria não foi encontrada no dataset."
    )

if dataset["categoria"].isna().any():
    raise ValueError(
        "A coluna categoria contém valores nulos."
    )

observed_categories = set(
    dataset["categoria"].astype(str).unique()
)
expected_categories = set(EXPECTED_CATEGORIES)

unexpected_categories = sorted(
    observed_categories - expected_categories
)
missing_categories = sorted(
    expected_categories - observed_categories
)

if unexpected_categories:
    raise ValueError(
        "Categorias inesperadas encontradas: "
        + ", ".join(unexpected_categories)
    )

if missing_categories:
    raise ValueError(
        "Categorias obrigatórias ausentes: "
        + ", ".join(missing_categories)
    )

category_counts = (
    dataset["categoria"]
    .value_counts()
    .reindex(EXPECTED_CATEGORIES)
    .astype(int)
)

category_percentages = (
    category_counts
    .div(len(dataset))
    .mul(100)
)

category_summary = pd.DataFrame(
    {
        "quantidade": category_counts,
        "percentual": category_percentages,
    }
)
category_summary.index.name = "categoria"

if int(category_summary["quantidade"].sum()) != len(dataset):
    raise RuntimeError(
        "A soma das categorias não corresponde "
        "ao total de registros."
    )

display(category_summary)

figure, axis = plt.subplots(figsize=(8, 4))
category_counts.plot(
    kind="bar",
    ax=axis,
)
axis.set_title("Frequência das categorias")
axis.set_xlabel("Categoria")
axis.set_ylabel("Quantidade")
axis.tick_params(axis="x", rotation=0)
axis.grid(axis="y", alpha=0.3)
figure.tight_layout()
plt.show()

print(
    "Distribuição validada para as três categorias. "
    "Os percentuais descrevem somente a base sintética."
)


### 4.2. Frequência por tipo de imóvel


In [ ]:
"""Analisa a frequência dos tipos de imóvel do Dataset EnergIAI V2."""

EXPECTED_PROPERTY_TYPES = [
    "APARTAMENTO",
    "CASA",
    "COMERCIO",
    "ESCRITORIO",
    "INDUSTRIA",
    "OUTRO",
]

if "tipo_imovel" not in dataset.columns:
    raise KeyError(
        "A coluna tipo_imovel não foi encontrada no dataset."
    )

if dataset["tipo_imovel"].isna().any():
    raise ValueError(
        "A coluna tipo_imovel contém valores nulos."
    )

observed_property_types = set(
    dataset["tipo_imovel"].astype(str).unique()
)
expected_property_types = set(EXPECTED_PROPERTY_TYPES)

unexpected_property_types = sorted(
    observed_property_types - expected_property_types
)
missing_property_types = sorted(
    expected_property_types - observed_property_types
)

if unexpected_property_types:
    raise ValueError(
        "Tipos de imóvel inesperados encontrados: "
        + ", ".join(unexpected_property_types)
    )

if missing_property_types:
    raise ValueError(
        "Tipos de imóvel obrigatórios ausentes: "
        + ", ".join(missing_property_types)
    )

property_type_counts = (
    dataset["tipo_imovel"]
    .value_counts()
    .reindex(EXPECTED_PROPERTY_TYPES)
    .astype(int)
)

property_type_percentages = (
    property_type_counts
    .div(len(dataset))
    .mul(100)
)

property_type_summary = pd.DataFrame(
    {
        "quantidade": property_type_counts,
        "percentual": property_type_percentages,
    }
)
property_type_summary.index.name = "tipo_imovel"

if int(property_type_summary["quantidade"].sum()) != len(dataset):
    raise RuntimeError(
        "A soma dos tipos de imóvel não corresponde "
        "ao total de registros."
    )

display(property_type_summary)

figure, axis = plt.subplots(figsize=(10, 4))
property_type_counts.plot(
    kind="bar",
    ax=axis,
)
axis.set_title("Frequência por tipo de imóvel")
axis.set_xlabel("Tipo de imóvel")
axis.set_ylabel("Quantidade")
axis.tick_params(axis="x", rotation=30)
axis.grid(axis="y", alpha=0.3)
figure.tight_layout()
plt.show()

print(
    "Distribuição validada para os seis tipos de imóvel. "
    "Os percentuais descrevem somente a base sintética."
)


### 4.3. Fronteiras, raros, outliers e cenários


In [ ]:
"""Analisa cenários, fronteiras, raros, outliers e quotas."""

import scenarios

AUDIT_COLUMNS = [
    "tipo_cenario",
    "caso_fronteira",
    "caso_raro",
    "outlier_plausivel",
    "score_referencia",
    "categoria",
    "tipo_imovel",
    "consumo_kwh",
    "quantidade_equipamentos",
    "horas_alto_consumo",
]

BOOLEAN_AUDIT_COLUMNS = [
    "caso_fronteira",
    "caso_raro",
    "outlier_plausivel",
]

NUMERIC_AUDIT_FEATURES = [
    "consumo_kwh",
    "quantidade_equipamentos",
    "horas_alto_consumo",
]

QUALITY_KEYS = {
    "record_count",
    "boundary_count",
    "rare_count",
    "outlier_count",
    "scenario_distribution",
}

missing_audit_columns = sorted(
    set(AUDIT_COLUMNS) - set(dataset.columns)
)

if missing_audit_columns:
    raise KeyError(
        "Colunas necessárias à auditoria ausentes: "
        + ", ".join(missing_audit_columns)
    )

missing_quality_keys = sorted(
    QUALITY_KEYS - set(quality_summary)
)

if missing_quality_keys:
    raise KeyError(
        "Indicadores ausentes no relatório de qualidade: "
        + ", ".join(missing_quality_keys)
    )

audit_data = dataset[AUDIT_COLUMNS].copy()

if audit_data.isna().any().any():
    null_columns = audit_data.columns[
        audit_data.isna().any()
    ].tolist()
    raise ValueError(
        "A auditoria contém valores nulos: "
        + ", ".join(null_columns)
    )

for column in BOOLEAN_AUDIT_COLUMNS:
    if not pd.api.types.is_bool_dtype(audit_data[column]):
        raise TypeError(
            f"{column} deve possuir tipo booleano."
        )

record_count = len(audit_data)

if record_count != int(quality_summary["record_count"]):
    raise ValueError(
        "A quantidade de registros difere do relatório "
        "de qualidade já validado."
    )

expected_boundary_count = int(
    round(record_count * scenarios.BOUNDARY_CASE_RATIO)
)
expected_rare_count = int(
    round(record_count * scenarios.RARE_CASE_RATIO)
)
expected_outlier_count = int(
    round(
        record_count
        * scenarios.PLAUSIBLE_OUTLIER_RATIO
    )
)
expected_typical_count = (
    record_count
    - expected_boundary_count
    - expected_rare_count
)

expected_scenario_counts = pd.Series(
    {
        "TIPICO": expected_typical_count,
        "FRONTEIRA": expected_boundary_count,
        "RARO_EXTREMO": expected_rare_count,
    },
    name="quantidade_esperada",
).reindex(scenarios.SCENARIO_TYPES)

observed_scenario_counts = (
    audit_data["tipo_cenario"]
    .value_counts()
    .reindex(
        scenarios.SCENARIO_TYPES,
        fill_value=0,
    )
    .rename("quantidade_observada")
)

validated_scenario_counts = pd.Series(
    quality_summary["scenario_distribution"],
    name="quantidade_validada",
).reindex(
    scenarios.SCENARIO_TYPES,
    fill_value=0,
)

scenario_summary = pd.concat(
    [
        observed_scenario_counts,
        validated_scenario_counts,
        expected_scenario_counts,
    ],
    axis=1,
)

scenario_summary["percentual_observado"] = (
    scenario_summary["quantidade_observada"]
    .div(record_count)
    .mul(100)
    .round(2)
)
scenario_summary["percentual_esperado"] = (
    scenario_summary["quantidade_esperada"]
    .div(record_count)
    .mul(100)
    .round(2)
)

if not scenario_summary[
    "quantidade_observada"
].equals(
    scenario_summary["quantidade_validada"]
):
    raise ValueError(
        "A contagem direta por cenário difere do relatório "
        "de qualidade já validado."
    )

if not scenario_summary[
    "quantidade_observada"
].equals(
    scenario_summary["quantidade_esperada"]
):
    raise ValueError(
        "As quantidades observadas por cenário diferem "
        "das quotas do contrato."
    )

display(scenario_summary)

figure, axis = plt.subplots(figsize=(8, 5))
scenario_summary["quantidade_observada"].plot(
    kind="bar",
    ax=axis,
)
axis.set_title("Quantidade de registros por cenário")
axis.set_xlabel("Cenário")
axis.set_ylabel("Quantidade")
axis.tick_params(axis="x", rotation=0)
figure.tight_layout()
plt.show()

boundary_flags = audit_data["caso_fronteira"]
rare_flags = audit_data["caso_raro"]
outlier_flags = audit_data["outlier_plausivel"]

quota_summary = pd.DataFrame(
    {
        "quantidade_observada": [
            int(boundary_flags.sum()),
            int(rare_flags.sum()),
            int(outlier_flags.sum()),
        ],
        "quantidade_validada": [
            int(quality_summary["boundary_count"]),
            int(quality_summary["rare_count"]),
            int(quality_summary["outlier_count"]),
        ],
        "quantidade_esperada": [
            expected_boundary_count,
            expected_rare_count,
            expected_outlier_count,
        ],
    },
    index=[
        "casos_fronteira",
        "casos_raros",
        "outliers_plausiveis",
    ],
)

quota_summary["percentual_observado"] = (
    quota_summary["quantidade_observada"]
    .div(record_count)
    .mul(100)
    .round(2)
)
quota_summary["percentual_esperado"] = (
    quota_summary["quantidade_esperada"]
    .div(record_count)
    .mul(100)
    .round(2)
)

if not quota_summary[
    "quantidade_observada"
].equals(
    quota_summary["quantidade_validada"]
):
    raise ValueError(
        "A contagem direta das flags difere do relatório "
        "de qualidade já validado."
    )

if not quota_summary[
    "quantidade_observada"
].equals(
    quota_summary["quantidade_esperada"]
):
    raise ValueError(
        "As quantidades observadas das flags diferem "
        "das quotas do contrato."
    )

display(quota_summary)

relationship_summary = pd.Series(
    {
        "fronteira_e_raro": int(
            (boundary_flags & rare_flags).sum()
        ),
        "outlier_fora_de_raro": int(
            (outlier_flags & ~rare_flags).sum()
        ),
        "outlier_e_fronteira": int(
            (outlier_flags & boundary_flags).sum()
        ),
        "fronteira_com_cenario_incorreto": int(
            (
                boundary_flags
                & ~audit_data["tipo_cenario"].eq(
                    "FRONTEIRA"
                )
            ).sum()
        ),
        "raro_com_cenario_incorreto": int(
            (
                rare_flags
                & ~audit_data["tipo_cenario"].eq(
                    "RARO_EXTREMO"
                )
            ).sum()
        ),
        "tipico_com_cenario_incorreto": int(
            (
                ~boundary_flags
                & ~rare_flags
                & ~audit_data["tipo_cenario"].eq(
                    "TIPICO"
                )
            ).sum()
        ),
    },
    name="quantidade_invalida",
)

if relationship_summary.ne(0).any():
    raise ValueError(
        "Foram encontradas inconsistências entre cenários "
        "e flags de auditoria."
    )

display(relationship_summary)

boundary_score_values = [
    scenarios.REFERENCE_SCORE_CATEGORY_RANGES[
        "EFICIENTE"
    ][1],
    scenarios.REFERENCE_SCORE_CATEGORY_RANGES[
        "MODERADO"
    ][0],
    scenarios.REFERENCE_SCORE_CATEGORY_RANGES[
        "MODERADO"
    ][1],
    scenarios.REFERENCE_SCORE_CATEGORY_RANGES[
        "INEFICIENTE"
    ][0],
]

boundary_rows = audit_data.loc[
    boundary_flags,
    [
        "score_referencia",
        "categoria",
        "tipo_imovel",
    ],
].copy()

unexpected_boundary_scores = sorted(
    set(boundary_rows["score_referencia"])
    - set(boundary_score_values)
)

if unexpected_boundary_scores:
    raise ValueError(
        "Casos marcados como fronteira possuem scores "
        "fora de 30, 31, 60 e 61: "
        + ", ".join(
            str(value)
            for value in unexpected_boundary_scores
        )
    )

boundary_score_summary = (
    boundary_rows
    .groupby(
        ["score_referencia", "categoria"],
        observed=True,
    )
    .size()
    .rename("quantidade")
    .to_frame()
)

display(boundary_score_summary)

outside_typical_counts = pd.Series(
    0,
    index=audit_data.index,
    dtype=int,
)

for property_type in sorted(
    audit_data["tipo_imovel"].astype(str).unique()
):
    if property_type not in scenarios.TYPICAL_RANGES:
        raise ValueError(
            "Tipo de imóvel sem faixa típica configurada: "
            f"{property_type}."
        )

    property_mask = audit_data[
        "tipo_imovel"
    ].eq(property_type)

    for feature in NUMERIC_AUDIT_FEATURES:
        minimum, maximum = scenarios.TYPICAL_RANGES[
            property_type
        ][feature]
        outside_feature_mask = property_mask & (
            audit_data[feature].lt(minimum)
            | audit_data[feature].gt(maximum)
        )
        outside_typical_counts.loc[
            outside_feature_mask
        ] += 1

range_consistency_summary = pd.Series(
    {
        "raros_sem_feature_fora_da_faixa": int(
            (
                rare_flags
                & outside_typical_counts.eq(0)
            ).sum()
        ),
        "nao_raros_com_feature_fora_da_faixa": int(
            (
                ~rare_flags
                & outside_typical_counts.gt(0)
            ).sum()
        ),
    },
    name="quantidade_invalida",
)

if range_consistency_summary.ne(0).any():
    raise ValueError(
        "Foram encontradas inconsistências entre a flag "
        "de caso raro e as faixas típicas."
    )

display(range_consistency_summary)

audit_group = pd.Series(
    "TIPICO",
    index=audit_data.index,
    dtype="object",
)
audit_group.loc[boundary_flags] = "FRONTEIRA"
audit_group.loc[
    rare_flags & ~outlier_flags
] = "RARO_SEM_OUTLIER"
audit_group.loc[
    outlier_flags
] = "OUTLIER_PLAUSIVEL"

scenario_numeric_summary = (
    audit_data
    .assign(grupo_auditoria=audit_group)
    .groupby(
        "grupo_auditoria",
        observed=True,
    )[NUMERIC_AUDIT_FEATURES]
    .agg(["min", "median", "max"])
    .round(2)
)

display(scenario_numeric_summary)

outside_features_by_group = pd.crosstab(
    audit_group.rename("grupo_auditoria"),
    outside_typical_counts.rename(
        "features_fora_da_faixa"
    ),
    normalize="index",
).mul(100).round(2)

display(outside_features_by_group)

category_by_scenario = (
    pd.crosstab(
        audit_data["tipo_cenario"],
        audit_data["categoria"],
        normalize="index",
    )
    .reindex(
        index=scenarios.SCENARIO_TYPES,
        fill_value=0.0,
    )
    .reindex(
        columns=[
            "EFICIENTE",
            "MODERADO",
            "INEFICIENTE",
        ],
        fill_value=0.0,
    )
    .mul(100)
    .round(2)
)

display(category_by_scenario)

property_by_scenario = (
    pd.crosstab(
        audit_data["tipo_cenario"],
        audit_data["tipo_imovel"],
        normalize="index",
    )
    .reindex(
        index=scenarios.SCENARIO_TYPES,
        fill_value=0.0,
    )
    .mul(100)
    .round(2)
)

display(property_by_scenario)

print(
    "As quotas foram comparadas com o relatório de "
    "qualidade e com os parâmetros do contrato."
)
print(
    "Os campos de auditoria são usados somente para "
    "análise e não podem entrar como features do modelo."
)
print(
    "As diferenças observadas descrevem a construção "
    "da base sintética e não representam padrões reais."
)


## 5. Divisão estratificada 70/15/15

O split é criado e validado exclusivamente pela função central
`data-science/src/data_split.py`.

O notebook somente consome os subconjuntos e apresenta seus tamanhos. As
validações de seed, schema, proporções, alinhamento, sobreposição, reconstrução,
cinco features, campos proibidos, nulos, finitude e classes válidas permanecem
centralizadas no módulo e em seus testes.


In [ ]:
"""Consome o split estratificado central 70/15/15."""

import data_split


split = data_split.create_stratified_data_split(
    dataset,
    seed=RANDOM_SEED,
)

X_train = split.x_train
X_validation = split.x_validation
X_test = split.x_test
y_train = split.y_train
y_validation = split.y_validation
y_test = split.y_test

PRODUCTION_FEATURES = list(X_train.columns)

split_summary = pd.DataFrame(
    {
        "quantidade": [
            len(X_train),
            len(X_validation),
            len(X_test),
        ],
        "percentual": [
            len(X_train) / len(dataset) * 100,
            len(X_validation) / len(dataset) * 100,
            len(X_test) / len(dataset) * 100,
        ],
    },
    index=[
        "treino",
        "validacao",
        "teste",
    ],
).round(2)

display(split_summary)

print(
    "Split central consumido: 3.500 registros de treino, "
    "750 de validação e 750 de teste."
)
print(
    "A integridade dos três subconjuntos foi validada em "
    "data_split.create_stratified_data_split."
)
print(
    "Nenhuma distribuição ou estatística descritiva do teste "
    "foi calculada no notebook."
)


### 5.1. Distribuições numéricas no treino

As distribuições numéricas são analisadas exclusivamente no conjunto de
treino. Campos de auditoria, target e derivados do target não participam desta
EDA.


In [ ]:
"""Analisa as distribuições numéricas somente no treino."""

NUMERIC_FEATURE_COLUMNS = [
    "consumo_kwh",
    "quantidade_equipamentos",
    "horas_alto_consumo",
]

missing_columns = [
    column
    for column in NUMERIC_FEATURE_COLUMNS
    if column not in X_train.columns
]

if missing_columns:
    raise KeyError(
        "Features numéricas ausentes no treino: "
        + ", ".join(missing_columns)
    )

non_numeric_columns = [
    column
    for column in NUMERIC_FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(X_train[column])
]

if non_numeric_columns:
    raise TypeError(
        "Features com tipo não numérico no treino: "
        + ", ".join(non_numeric_columns)
    )

numeric_eda = X_train.loc[
    :,
    NUMERIC_FEATURE_COLUMNS,
].copy()

if numeric_eda.isna().any().any():
    raise ValueError(
        "A EDA numérica do treino encontrou valores nulos."
    )

finite_values = np.isfinite(
    numeric_eda.to_numpy(dtype=float)
)

if not finite_values.all():
    raise ValueError(
        "A EDA numérica do treino encontrou valores não finitos."
    )

numeric_summary = numeric_eda.describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
).T

numeric_summary["iqr"] = (
    numeric_eda.quantile(0.75)
    - numeric_eda.quantile(0.25)
)
numeric_summary["assimetria"] = numeric_eda.skew()

display(numeric_summary)

for column in NUMERIC_FEATURE_COLUMNS:
    title = column.replace("_", " ").title()

    figure, axis = plt.subplots(figsize=(8, 4))
    axis.hist(
        numeric_eda[column],
        bins=30,
    )
    axis.set_title(f"Distribuição de {title} no treino")
    axis.set_xlabel(title)
    axis.set_ylabel("Frequência")
    axis.grid(axis="y", alpha=0.3)
    figure.tight_layout()
    plt.show()

    figure, axis = plt.subplots(figsize=(8, 2.5))
    axis.boxplot(
        numeric_eda[column],
        orientation="horizontal",
    )
    axis.set_title(f"Boxplot de {title} no treino")
    axis.set_xlabel(title)
    axis.grid(axis="x", alpha=0.3)
    figure.tight_layout()
    plt.show()

print(
    "A EDA numérica utilizou somente as três features numéricas "
    "do conjunto de treino."
)


### 5.2. Relações entre features no treino

Correlações, associações e resumos por tipo de imóvel são calculados somente
sobre as cinco features de produção do conjunto de treino.


In [ ]:
"""Analisa relações entre as cinco features somente no treino."""

from scipy.stats import chi2_contingency


CONTINUOUS_RELATION_FEATURES = [
    "consumo_kwh",
    "quantidade_equipamentos",
    "horas_alto_consumo",
]

CORRELATION_FEATURES = [
    "consumo_kwh",
    "uso_horario_pico",
    "quantidade_equipamentos",
    "horas_alto_consumo",
]

EXCLUDED_ANALYSIS_COLUMNS = [
    "categoria",
    "score_referencia",
    "tipo_cenario",
    "caso_fronteira",
    "caso_raro",
    "outlier_plausivel",
    "lote_geracao",
]

missing_features = sorted(
    set(PRODUCTION_FEATURES) - set(X_train.columns)
)

if missing_features:
    raise KeyError(
        "Features de produção ausentes: "
        + ", ".join(missing_features)
    )

relation_data = X_train.loc[
    :,
    PRODUCTION_FEATURES,
].copy()

leaked_columns = sorted(
    set(relation_data.columns)
    & set(EXCLUDED_ANALYSIS_COLUMNS)
)

if leaked_columns:
    raise RuntimeError(
        "Colunas proibidas entraram na análise: "
        + ", ".join(leaked_columns)
    )

if relation_data.isna().any().any():
    null_columns = relation_data.columns[
        relation_data.isna().any()
    ].tolist()
    raise ValueError(
        "As features de produção contêm valores nulos: "
        + ", ".join(null_columns)
    )

peak_values = relation_data["uso_horario_pico"]

if pd.api.types.is_bool_dtype(peak_values):
    relation_data["uso_horario_pico"] = peak_values.astype(int)
else:
    normalized_peak_values = (
        peak_values
        .astype(str)
        .str.strip()
        .str.lower()
    )
    peak_mapping = {
        "false": 0,
        "true": 1,
        "0": 0,
        "1": 1,
    }
    unexpected_peak_values = sorted(
        set(normalized_peak_values) - set(peak_mapping)
    )

    if unexpected_peak_values:
        raise ValueError(
            "Valores inesperados em uso_horario_pico: "
            + ", ".join(unexpected_peak_values)
        )

    relation_data["uso_horario_pico"] = (
        normalized_peak_values
        .map(peak_mapping)
        .astype(int)
    )

for feature in CORRELATION_FEATURES:
    relation_data[feature] = pd.to_numeric(
        relation_data[feature],
        errors="raise",
    )

numeric_values = relation_data[
    CORRELATION_FEATURES
].to_numpy(dtype=float)

if not np.isfinite(numeric_values).all():
    raise ValueError(
        "As features numéricas contêm valores não finitos."
    )

correlation_matrix = relation_data[
    CORRELATION_FEATURES
].corr(method="pearson")

if correlation_matrix.isna().any().any():
    raise RuntimeError(
        "A matriz de correlação contém valores ausentes."
    )

display(correlation_matrix.round(3))

figure, axis = plt.subplots(figsize=(9, 6))
image = axis.imshow(
    correlation_matrix.to_numpy(),
    vmin=-1,
    vmax=1,
)
axis.set_title(
    "Correlação de Pearson entre features numéricas"
)
axis.set_xticks(
    range(len(CORRELATION_FEATURES))
)
axis.set_xticklabels(
    CORRELATION_FEATURES,
    rotation=30,
    ha="right",
)
axis.set_yticks(
    range(len(CORRELATION_FEATURES))
)
axis.set_yticklabels(CORRELATION_FEATURES)

for row_index in range(len(CORRELATION_FEATURES)):
    for column_index in range(
        len(CORRELATION_FEATURES)
    ):
        axis.text(
            column_index,
            row_index,
            f"{correlation_matrix.iloc[row_index, column_index]:.2f}",
            ha="center",
            va="center",
        )

figure.colorbar(
    image,
    ax=axis,
    label="Correlação de Pearson",
)
figure.tight_layout()
plt.show()

property_numeric_summary = (
    relation_data
    .groupby("tipo_imovel", observed=True)[
        CONTINUOUS_RELATION_FEATURES
    ]
    .agg(["mean", "median"])
    .round(3)
)

display(property_numeric_summary)

peak_by_property = (
    pd.crosstab(
        relation_data["tipo_imovel"],
        relation_data["uso_horario_pico"],
        normalize="index",
    )
    .reindex(columns=[0, 1], fill_value=0.0)
    .rename(
        columns={
            0: "fora_horario_pico",
            1: "uso_horario_pico",
        }
    )
    .mul(100)
    .round(2)
)

display(peak_by_property)

contingency_table = pd.crosstab(
    relation_data["tipo_imovel"],
    relation_data["uso_horario_pico"],
)

if min(contingency_table.shape) <= 1:
    raise RuntimeError(
        "A tabela de contingência não possui dimensões "
        "suficientes para calcular o V de Cramér."
    )

chi_square, _, _, _ = chi2_contingency(
    contingency_table
)
cramers_denominator = (
    len(relation_data)
    * min(
        contingency_table.shape[0] - 1,
        contingency_table.shape[1] - 1,
    )
)

if cramers_denominator <= 0:
    raise RuntimeError(
        "O denominador do V de Cramér deve ser positivo."
    )

cramers_v = float(
    np.sqrt(chi_square / cramers_denominator)
)

association_summary = pd.Series(
    {
        "v_cramer_tipo_imovel_uso_horario_pico": (
            cramers_v
        ),
    },
    name="valor",
).round(4)

display(association_summary)

print(
    "Foram analisadas somente as cinco features de produção. "
    "Target, score e campos de auditoria foram excluídos."
)
print(
    "Para uso_horario_pico em 0/1, a correlação de Pearson "
    "equivale à correlação ponto-bisserial."
)
print(
    "As correlações e associações descrevem a base sintética "
    "e não representam causalidade ou validade externa."
)


### 5.3. PCA e K-Means auxiliares no treino

O pré-processamento exploratório, o PCA e o K-Means são ajustados
exclusivamente no conjunto de treino. Os clusters não são persistidos nem
utilizados como target ou feature de produção.


In [ ]:
"""Executa PCA e K-Means auxiliares somente no treino."""

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CONTINUOUS_AUXILIARY_FEATURES = list(NUMERIC_FEATURE_COLUMNS)
BINARY_AUXILIARY_FEATURES = ["uso_horario_pico"]
CATEGORICAL_AUXILIARY_FEATURES = ["tipo_imovel"]
K_VALUES = list(range(2, 9))
SILHOUETTE_SAMPLE_SIZE = min(2_000, len(relation_data))

auxiliary_data = relation_data.loc[
    :,
    PRODUCTION_FEATURES,
].copy()

if list(auxiliary_data.columns) != PRODUCTION_FEATURES:
    raise RuntimeError(
        "A análise auxiliar deve receber exatamente "
        "as cinco features de produção."
    )

auxiliary_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            CONTINUOUS_AUXILIARY_FEATURES,
        ),
        (
            "binary",
            StandardScaler(),
            BINARY_AUXILIARY_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_AUXILIARY_FEATURES,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

auxiliary_matrix = auxiliary_preprocessor.fit_transform(
    auxiliary_data
)
auxiliary_matrix = np.asarray(
    auxiliary_matrix,
    dtype=float,
)
transformed_feature_names = (
    auxiliary_preprocessor.get_feature_names_out()
)

if auxiliary_matrix.shape[0] != len(auxiliary_data):
    raise RuntimeError(
        "A matriz auxiliar possui quantidade de registros inválida."
    )

if auxiliary_matrix.shape[1] != len(
    transformed_feature_names
):
    raise RuntimeError(
        "A quantidade de colunas transformadas difere "
        "dos nomes produzidos pelo pré-processamento."
    )

if not np.isfinite(auxiliary_matrix).all():
    raise ValueError(
        "A matriz auxiliar contém valores não finitos."
    )

transformation_summary = pd.Series(
    {
        "registros": auxiliary_matrix.shape[0],
        "features_originais": len(PRODUCTION_FEATURES),
        "colunas_transformadas": auxiliary_matrix.shape[1],
        "amostra_silhouette": SILHOUETTE_SAMPLE_SIZE,
    },
    name="valor",
)

display(transformation_summary)
display(
    pd.Series(
        transformed_feature_names,
        name="coluna_transformada",
    )
)

pca_model = PCA(svd_solver="full")
pca_coordinates = pca_model.fit_transform(
    auxiliary_matrix
)

pca_summary = pd.DataFrame(
    {
        "componente": np.arange(
            1,
            len(pca_model.explained_variance_ratio_) + 1,
        ),
        "variancia_explicada": (
            pca_model.explained_variance_ratio_
        ),
        "variancia_acumulada": np.cumsum(
            pca_model.explained_variance_ratio_
        ),
    }
)

if not np.isclose(
    pca_summary["variancia_explicada"].sum(),
    1.0,
):
    raise RuntimeError(
        "A variância explicada pelo PCA não soma 1."
    )

display(pca_summary.round(4))

figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(
    pca_summary["componente"],
    pca_summary["variancia_acumulada"],
    marker="o",
)
axis.axhline(
    0.80,
    linestyle="--",
    label="80% da variância",
)
axis.set_title("Variância acumulada por componente principal")
axis.set_xlabel("Componente principal")
axis.set_ylabel("Variância explicada acumulada")
axis.set_ylim(0.0, 1.05)
axis.set_xticks(pca_summary["componente"])
axis.grid(alpha=0.3)
axis.legend()
figure.tight_layout()
plt.show()

cluster_models = {}
cluster_metric_rows = []

for cluster_count in K_VALUES:
    cluster_model = KMeans(
        n_clusters=cluster_count,
        random_state=RANDOM_SEED,
        n_init=20,
    )
    cluster_labels = cluster_model.fit_predict(
        auxiliary_matrix
    )
    silhouette = silhouette_score(
        auxiliary_matrix,
        cluster_labels,
        sample_size=SILHOUETTE_SAMPLE_SIZE,
        random_state=RANDOM_SEED,
    )

    cluster_models[cluster_count] = cluster_model
    cluster_metric_rows.append(
        {
            "k": cluster_count,
            "inercia": cluster_model.inertia_,
            "silhouette": silhouette,
        }
    )

cluster_metrics = pd.DataFrame(cluster_metric_rows)

if cluster_metrics.isna().any().any():
    raise RuntimeError(
        "As métricas auxiliares de clusterização contêm nulos."
    )

display(cluster_metrics.round(4))

figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(
    cluster_metrics["k"],
    cluster_metrics["inercia"],
    marker="o",
)
axis.set_title("Método do cotovelo para K-Means")
axis.set_xlabel("Quantidade de clusters (k)")
axis.set_ylabel("Inércia")
axis.set_xticks(K_VALUES)
axis.grid(alpha=0.3)
figure.tight_layout()
plt.show()

figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(
    cluster_metrics["k"],
    cluster_metrics["silhouette"],
    marker="o",
)
axis.set_title("Silhouette por quantidade de clusters")
axis.set_xlabel("Quantidade de clusters (k)")
axis.set_ylabel("Silhouette")
axis.set_xticks(K_VALUES)
axis.grid(alpha=0.3)
figure.tight_layout()
plt.show()

best_k = int(
    cluster_metrics.loc[
        cluster_metrics["silhouette"].idxmax(),
        "k",
    ]
)
best_cluster_model = cluster_models[best_k]
best_cluster_labels = pd.Series(
    best_cluster_model.labels_,
    index=auxiliary_data.index,
    name="cluster_auxiliar",
)

cluster_summary = pd.DataFrame(
    {
        "quantidade": (
            best_cluster_labels
            .value_counts()
            .sort_index()
        ),
    }
)
cluster_summary["percentual"] = (
    cluster_summary["quantidade"]
    .div(len(auxiliary_data))
    .mul(100)
    .round(2)
)
cluster_summary.index.name = "cluster_auxiliar"

display(cluster_summary)

cluster_profile_data = auxiliary_data.assign(
    cluster_auxiliar=best_cluster_labels
)
cluster_numeric_profile = (
    cluster_profile_data
    .groupby(
        "cluster_auxiliar",
        observed=True,
    )[CONTINUOUS_AUXILIARY_FEATURES]
    .median()
    .round(2)
)
cluster_numeric_profile[
    "uso_horario_pico_percentual"
] = (
    cluster_profile_data
    .groupby(
        "cluster_auxiliar",
        observed=True,
    )["uso_horario_pico"]
    .mean()
    .mul(100)
    .round(2)
)

display(cluster_numeric_profile)

property_distribution_by_cluster = (
    pd.crosstab(
        best_cluster_labels,
        auxiliary_data["tipo_imovel"],
        normalize="index",
    )
    .mul(100)
    .round(2)
)

display(property_distribution_by_cluster)

figure, axis = plt.subplots(figsize=(8, 6))
scatter = axis.scatter(
    pca_coordinates[:, 0],
    pca_coordinates[:, 1],
    c=best_cluster_labels.to_numpy(),
    s=12,
    alpha=0.55,
)
axis.set_title(
    f"PCA em duas dimensões com K-Means auxiliar (k={best_k})"
)
axis.set_xlabel(
    "Componente principal 1 "
    f"({pca_model.explained_variance_ratio_[0]:.1%})"
)
axis.set_ylabel(
    "Componente principal 2 "
    f"({pca_model.explained_variance_ratio_[1]:.1%})"
)
axis.grid(alpha=0.2)
axis.legend(
    *scatter.legend_elements(),
    title="Cluster auxiliar",
)
figure.tight_layout()
plt.show()

if "cluster_auxiliar" in dataset.columns:
    raise RuntimeError(
        "O cluster auxiliar não pode ser persistido no dataset."
    )

print(
    f"Melhor silhouette exploratório no intervalo testado: k={best_k}."
)
print(
    "O silhouette foi calculado em amostra determinística de "
    f"{SILHOUETTE_SAMPLE_SIZE} registros; PCA e K-Means foram "
    f"ajustados somente sobre {len(auxiliary_data)} registros "
    "de treino."
)
print(
    "PCA e K-Means são diagnósticos auxiliares. Os clusters não "
    "são target, feature de produção ou campo da API."
)
print(
    "A codificação one-hot de tipo_imovel torna o PCA uma "
    "aproximação exploratória para dados mistos."
)
print(
    "O resultado mede a capacidade do modelo de reproduzir padrões "
    "da base sintética sob as condições testadas."
)
print(
    "Os agrupamentos descrevem somente a base sintética e não "
    "representam segmentos reais, causalidade ou validade externa."
)


### 5.4. Comparação entre treino e validação

As distribuições e estatísticas descritivas são comparadas somente entre
treino e validação.

O holdout não é consultado nesta análise. Seus checks automáticos permitidos
são executados pelo contrato central de `data_split.py` e pelos testes
consolidados.


In [ ]:
"""Compara treino e validação sem consultar o holdout."""

MODELING_SPLIT_NAMES = [
    "treino",
    "validacao",
]

MODELING_SPLIT_FEATURES = {
    "treino": X_train,
    "validacao": X_validation,
}

MODELING_SPLIT_TARGETS = {
    "treino": y_train,
    "validacao": y_validation,
}

category_distribution_modeling = pd.concat(
    {
        split_name: (
            MODELING_SPLIT_TARGETS[split_name]
            .value_counts(normalize=True)
            .reindex(EXPECTED_CATEGORIES, fill_value=0.0)
            .mul(100)
        )
        for split_name in MODELING_SPLIT_NAMES
    },
    axis=1,
).round(2)

category_distribution_modeling.index.name = "categoria"

display(category_distribution_modeling)

property_distribution_modeling = pd.concat(
    {
        split_name: (
            MODELING_SPLIT_FEATURES[split_name]["tipo_imovel"]
            .astype(str)
            .value_counts(normalize=True)
            .reindex(
                EXPECTED_PROPERTY_TYPES,
                fill_value=0.0,
            )
            .mul(100)
        )
        for split_name in MODELING_SPLIT_NAMES
    },
    axis=1,
).round(2)

property_distribution_modeling.index.name = "tipo_imovel"

display(property_distribution_modeling)

numeric_summary_rows = []

for split_name in MODELING_SPLIT_NAMES:
    split_numeric_data = MODELING_SPLIT_FEATURES[
        split_name
    ][NUMERIC_FEATURE_COLUMNS]

    for feature in NUMERIC_FEATURE_COLUMNS:
        numeric_summary_rows.append(
            {
                "split": split_name,
                "feature": feature,
                "media": split_numeric_data[feature].mean(),
                "mediana": split_numeric_data[feature].median(),
                "desvio_padrao": split_numeric_data[feature].std(),
                "minimo": split_numeric_data[feature].min(),
                "maximo": split_numeric_data[feature].max(),
            }
        )

numeric_summary_modeling = (
    pd.DataFrame(numeric_summary_rows)
    .set_index(["feature", "split"])
    .sort_index()
    .round(2)
)

display(numeric_summary_modeling)

peak_distribution_modeling = pd.concat(
    {
        split_name: (
            MODELING_SPLIT_FEATURES[
                split_name
            ]["uso_horario_pico"]
            .astype(str)
            .str.strip()
            .str.lower()
            .value_counts(normalize=True)
            .reindex(["false", "true"], fill_value=0.0)
            .mul(100)
        )
        for split_name in MODELING_SPLIT_NAMES
    },
    axis=1,
).round(2)

peak_distribution_modeling.index.name = "uso_horario_pico"

display(peak_distribution_modeling)

for comparison_name, comparison_frame in {
    "categorias": category_distribution_modeling,
    "tipos de imóvel": property_distribution_modeling,
    "variáveis numéricas": numeric_summary_modeling,
    "uso em horário de pico": peak_distribution_modeling,
}.items():
    if comparison_frame.isna().any().any():
        raise RuntimeError(
            f"A comparação de {comparison_name} contém valores nulos."
        )

if not np.allclose(
    category_distribution_modeling.sum(axis=0).to_numpy(),
    np.full(len(MODELING_SPLIT_NAMES), 100.0),
):
    raise RuntimeError(
        "Os percentuais de categoria não somam 100% "
        "em treino e validação."
    )

if not np.allclose(
    property_distribution_modeling.sum(axis=0).to_numpy(),
    np.full(len(MODELING_SPLIT_NAMES), 100.0),
    atol=0.05,
):
    raise RuntimeError(
        "Os percentuais de tipo de imóvel não somam "
        "aproximadamente 100% em treino e validação."
    )

if not np.allclose(
    peak_distribution_modeling.sum(axis=0).to_numpy(),
    np.full(len(MODELING_SPLIT_NAMES), 100.0),
    atol=0.05,
):
    raise RuntimeError(
        "Os percentuais de uso em horário de pico não somam "
        "aproximadamente 100% em treino e validação."
    )

print(
    "Treino e validação foram comparados por distribuições "
    "e estatísticas descritivas."
)
print(
    "O holdout não foi consultado nesta análise."
)


## 6. Validação da contribuição multivariada

Esta seção avalia a contribuição individual e conjunta das cinco features de
produção. A mutual information é calculada exclusivamente sobre o conjunto de
treino. Os modelos individuais são ajustados no treino e avaliados na
validação.

O conjunto de teste permanece isolado. Target, score de referência e campos de
auditoria não participam dos diagnósticos.


In [ ]:
"""Avalia mutual information e modelos com uma feature por vez."""

import baseline_benchmark
import multivariate_validation


GUARDRAIL_RATIO_LIMIT = 0.95

mutual_information_results = (
    multivariate_validation.calculate_mutual_information(
        X_train,
        y_train,
        seed=RANDOM_SEED,
    )
)

single_feature_results = (
    multivariate_validation.evaluate_single_feature_logistic(
        X_train,
        y_train,
        X_validation,
        y_validation,
        seed=RANDOM_SEED,
    )
)

full_logistic_score = baseline_benchmark.evaluate_logistic_baseline(
    X_train,
    y_train,
    X_validation,
    y_validation,
    seed=RANDOM_SEED,
)

expected_feature_order = tuple(PRODUCTION_FEATURES)

if tuple(mutual_information_results) != expected_feature_order:
    raise RuntimeError(
        "A mutual information não retornou as cinco features "
        "na ordem oficial."
    )

if tuple(single_feature_results) != expected_feature_order:
    raise RuntimeError(
        "Os modelos individuais não retornaram as cinco features "
        "na ordem oficial."
    )

if not np.isfinite(full_logistic_score):
    raise RuntimeError(
        "O F1-macro do modelo completo não é finito."
    )

if not 0.0 <= full_logistic_score <= 1.0:
    raise RuntimeError(
        "O F1-macro do modelo completo está fora do intervalo [0, 1]."
    )

mutual_information_summary = pd.Series(
    mutual_information_results,
    name="mutual_information_treino",
).reindex(PRODUCTION_FEATURES)

single_feature_summary = pd.Series(
    single_feature_results,
    name="f1_macro_individual_validacao",
).reindex(PRODUCTION_FEATURES)

multivariate_contribution_summary = pd.concat(
    [
        mutual_information_summary,
        single_feature_summary,
    ],
    axis=1,
)

multivariate_contribution_summary[
    "relacao_individual_modelo_completo"
] = (
    multivariate_contribution_summary[
        "f1_macro_individual_validacao"
    ]
    .div(full_logistic_score)
)

if multivariate_contribution_summary.isna().any().any():
    raise RuntimeError(
        "O resumo da contribuição multivariada contém valores nulos."
    )

if not np.isfinite(
    multivariate_contribution_summary.to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "O resumo da contribuição multivariada contém valores "
        "não finitos."
    )

best_single_feature = str(single_feature_summary.idxmax())
best_single_feature_score = float(
    single_feature_summary.loc[best_single_feature]
)
best_single_feature_ratio = (
    best_single_feature_score / full_logistic_score
)
guardrail_satisfied = (
    best_single_feature_ratio <= GUARDRAIL_RATIO_LIMIT
)

guardrail_summary = pd.Series(
    {
        "modelo_completo_f1_macro": full_logistic_score,
        "melhor_feature_individual": best_single_feature,
        "melhor_feature_individual_f1_macro": (
            best_single_feature_score
        ),
        "relacao_melhor_individual_completo": (
            best_single_feature_ratio
        ),
        "limite_guardrail": GUARDRAIL_RATIO_LIMIT,
        "guardrail_atendido": guardrail_satisfied,
    },
    name="valor",
)

if not guardrail_satisfied:
    raise RuntimeError(
        "O melhor modelo individual ultrapassou o limite de 95% "
        "do F1-macro do modelo completo."
    )

display(multivariate_contribution_summary.round(6))
display(guardrail_summary.to_frame())

figure, axis = plt.subplots(figsize=(9, 5))
single_feature_summary.plot(
    kind="bar",
    ax=axis,
)
axis.axhline(
    full_logistic_score,
    linestyle="--",
    label="Modelo completo",
)
axis.set_title(
    "F1-macro dos modelos individuais na validação"
)
axis.set_xlabel("Feature utilizada")
axis.set_ylabel("F1-macro")
axis.set_ylim(0.0, 1.0)
axis.tick_params(axis="x", rotation=35)
axis.grid(axis="y", alpha=0.3)
axis.legend()
figure.tight_layout()
plt.show()

print(
    "Melhor feature individual:",
    best_single_feature,
    f"({best_single_feature_score:.6f}).",
)
print(
    "Relação com o modelo completo:",
    f"{best_single_feature_ratio:.1%}.",
)
print(
    "Guardrail de 95% atendido:",
    guardrail_satisfied,
)
print(
    "Mutual information individual não deve ser usada isoladamente "
    "para excluir uma feature, pois não representa todas as interações."
)
print(
    "O resultado mede a capacidade do modelo de reproduzir padrões "
    "da base sintética sob as condições testadas."
)


### 6.1. Ablação e permutation importance

A ablação remove uma feature por execução e compara o F1-macro com o modelo
completo. A permutation importance é calculada no conjunto de validação após o
ajuste do pipeline no treino.

Esses diagnósticos complementam a mutual information individual. Nenhum deles
demonstra causalidade, validade externa ou desempenho em dados reais.


In [ ]:
"""Executa ablação e permutation importance com splits explícitos."""

PERMUTATION_REPEATS = 10

ablation_results = (
    multivariate_validation.evaluate_leave_one_feature_out_logistic(
        X_train,
        y_train,
        X_validation,
        y_validation,
        seed=RANDOM_SEED,
    )
)

permutation_results = (
    multivariate_validation.calculate_permutation_importance(
        X_train,
        y_train,
        X_validation,
        y_validation,
        seed=RANDOM_SEED,
        n_repeats=PERMUTATION_REPEATS,
    )
)

if tuple(ablation_results) != tuple(PRODUCTION_FEATURES):
    raise RuntimeError(
        "A ablação não retornou as cinco features na ordem oficial."
    )

if tuple(permutation_results) != tuple(PRODUCTION_FEATURES):
    raise RuntimeError(
        "A permutation importance não retornou as cinco features "
        "na ordem oficial."
    )

ablation_summary = pd.Series(
    ablation_results,
    name="f1_macro_sem_feature",
).reindex(PRODUCTION_FEATURES)

ablation_comparison = ablation_summary.to_frame()
ablation_comparison["f1_macro_modelo_completo"] = (
    full_logistic_score
)
ablation_comparison["queda_absoluta"] = (
    full_logistic_score
    - ablation_comparison["f1_macro_sem_feature"]
)
ablation_comparison.index.name = "feature_removida"

permutation_summary = pd.DataFrame.from_dict(
    permutation_results,
    orient="index",
).reindex(PRODUCTION_FEATURES)
permutation_summary.index.name = "feature_permutada"

if ablation_comparison.isna().any().any():
    raise RuntimeError(
        "O resumo de ablação contém valores nulos."
    )

if permutation_summary.isna().any().any():
    raise RuntimeError(
        "O resumo de permutation importance contém valores nulos."
    )

if not np.isfinite(
    ablation_comparison.to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "O resumo de ablação contém valores não finitos."
    )

if not np.isfinite(
    permutation_summary.to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "O resumo de permutation importance contém valores "
        "não finitos."
    )

display(ablation_comparison.round(6))
display(permutation_summary.round(6))

all_ablation_drops_positive = bool(
    ablation_comparison["queda_absoluta"].gt(0.0).all()
)
all_permutation_means_positive = bool(
    permutation_summary["importance_mean"].gt(0.0).all()
)

figure, axis = plt.subplots(figsize=(9, 5))
ablation_comparison["queda_absoluta"].plot(
    kind="bar",
    ax=axis,
)
axis.axhline(0.0, linestyle="--")
axis.set_title(
    "Queda de F1-macro ao remover cada feature"
)
axis.set_xlabel("Feature removida")
axis.set_ylabel("Queda absoluta de F1-macro")
axis.tick_params(axis="x", rotation=35)
axis.grid(axis="y", alpha=0.3)
figure.tight_layout()
plt.show()

figure, axis = plt.subplots(figsize=(9, 5))
permutation_summary["importance_mean"].plot(
    kind="bar",
    yerr=permutation_summary["importance_std"],
    ax=axis,
    capsize=4,
)
axis.axhline(0.0, linestyle="--")
axis.set_title(
    "Permutation importance na validação"
)
axis.set_xlabel("Feature permutada")
axis.set_ylabel("Queda média de F1-macro")
axis.tick_params(axis="x", rotation=35)
axis.grid(axis="y", alpha=0.3)
figure.tight_layout()
plt.show()

print(
    "Todas as remoções reduziram o F1-macro:",
    all_ablation_drops_positive,
)
print(
    "Todas as importâncias médias por permutação foram positivas:",
    all_permutation_means_positive,
)
print(
    "O conjunto de teste permaneceu isolado dos diagnósticos."
)
print(
    "O resultado mede a capacidade do modelo de reproduzir padrões "
    "da base sintética sob as condições testadas."
)
print(
    "Os resultados não demonstram causalidade, validade externa "
    "ou desempenho em dados reais."
)


## 7. Pré-processamento e pipelines

In [ ]:
# Implementar ColumnTransformer e pipelines sem vazamento.

## 8. Baselines e modelos individuais

In [ ]:
# Treinar Dummy, Regressão Logística, Árvore, Random Forest e HistGradientBoosting.

## 9. Busca controlada de hiperparâmetros

In [ ]:
# Implementar RandomizedSearchCV com validação estratificada.

## 10. Avaliação final

In [ ]:
# Calcular métricas, matriz de confusão, log loss e inferência.

### 10.1. Calibração

In [ ]:
# Comparar modelo original, sigmoid e isotonic quando aplicável.

## 11. Robustez e auditoria humana

In [ ]:
# Implementar testes de fronteira, extremos, seeds e amostra auditável.

## 12. Seleção e serialização do pipeline

In [ ]:
# Serializar o pipeline somente após seleção baseada em evidências.

## 13. Exemplos para integração FastAPI

In [ ]:
# Gerar três exemplos JSON compatíveis com POST /predict.

## 14. Conclusões e limitações

> O resultado mede a capacidade do modelo de reproduzir padrões da base sintética sob as condições testadas.

Nenhuma conclusão deste notebook representa desempenho em dados reais ou comprovação de integração OCI.